# Module 5.2: Markov-Switching Autoregressive Models

## Learning Objectives
By completing this notebook, you will:
1. Understand MS-AR model structure and motivation
2. Implement regime-dependent AR processes
3. Estimate parameters using EM algorithm
4. Apply to financial time series with regime dynamics
5. Compare MS-AR to standard HMM and AR models

## Prerequisites
- Standard HMM (Module 2)
- Autoregressive models basics

## Estimated Time: 60 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. MS-AR Model Definition

**Standard AR(p)**:
$$y_t = \phi_0 + \phi_1 y_{t-1} + ... + \phi_p y_{t-p} + \epsilon_t$$

**Markov-Switching AR(p)**:
$$y_t = \phi_{0,s_t} + \phi_{1,s_t} y_{t-1} + ... + \phi_{p,s_t} y_{t-p} + \epsilon_t$$

where $s_t$ is the hidden regime and $\epsilon_t \sim N(0, \sigma_{s_t}^2)$

### Key Features
- Regime-dependent autoregressive coefficients
- Regime-dependent intercepts and variance
- Captures changing persistence and mean reversion

In [ ]:
def generate_msar_data(T=1000, p=1):
    # 2-state MS-AR(1)
    A = np.array([[0.95, 0.05], [0.10, 0.90]])
    pi = np.array([0.6, 0.4])
    
    # State 0: Persistent (high phi)
    # State 1: Mean-reverting (low phi)
    params = [
        {'phi0': 0.0005, 'phi1': 0.90, 'sigma': 0.01},  # State 0
        {'phi0': 0.0002, 'phi1': 0.30, 'sigma': 0.015}  # State 1
    ]
    
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(2, p=pi)
    
    y = np.zeros(T)
    y[0] = 0.0
    
    for t in range(1, T):
        states[t] = np.random.choice(2, p=A[states[t-1]])
        s = states[t]
        y[t] = params[s]['phi0'] + params[s]['phi1'] * y[t-1] + \
               np.random.normal(0, params[s]['sigma'])
    
    return y, states

y, true_states = generate_msar_data(1000)

print('MS-AR(1) Data Generated')
print(f'Shape: {y.shape}')
print(f'Mean: {y.mean():.6f}')
print(f'Std: {y.std():.6f}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(y, linewidth=0.8)
axes[0].set_ylabel('Value')
axes[0].set_title('MS-AR(1) Time Series')
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(range(len(true_states)), true_states, alpha=0.7, step='mid')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Regime')
axes[1].set_title('True Regime Sequence')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Persistent', 'Mean-Reverting'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. MS-AR Model Implementation

In [ ]:
class MSAR:
    def __init__(self, n_states, ar_order):
        self.K = n_states
        self.p = ar_order
        
        # HMM parameters
        self.pi = np.ones(self.K) / self.K
        self.A = np.random.rand(self.K, self.K)
        self.A /= self.A.sum(axis=1, keepdims=True)
        
        # AR parameters (one set per state)
        self.phi0 = np.random.randn(self.K) * 0.001
        self.phi = np.random.randn(self.K, self.p) * 0.1
        self.sigma = np.ones(self.K) * 0.01
    
    def _emission_prob(self, y_t, y_lag, state):
        # P(y_t | y_{t-1}, ..., y_{t-p}, state)
        mean = self.phi0[state] + (self.phi[state] @ y_lag)
        return stats.norm.pdf(y_t, mean, self.sigma[state])
    
    def fit(self, y, max_iter=50, verbose=True):
        T = len(y)
        
        # Create lagged observations
        y_lagged = np.zeros((T-self.p, self.p))
        for i in range(self.p):
            y_lagged[:, i] = y[self.p-i-1:T-i-1]
        y_current = y[self.p:]
        T_eff = len(y_current)
        
        for iteration in range(max_iter):
            # E-step: Forward-Backward
            alpha = np.zeros((T_eff, self.K))
            scaling = np.zeros(T_eff)
            
            # Initialize
            for k in range(self.K):
                alpha[0, k] = self.pi[k] * self._emission_prob(y_current[0], y_lagged[0], k)
            scaling[0] = alpha[0].sum()
            alpha[0] /= scaling[0]
            
            # Forward pass
            for t in range(1, T_eff):
                for j in range(self.K):
                    alpha[t, j] = (alpha[t-1] @ self.A[:, j]) * \
                                  self._emission_prob(y_current[t], y_lagged[t], j)
                scaling[t] = alpha[t].sum()
                alpha[t] /= scaling[t]
            
            log_lik = np.log(scaling + 1e-300).sum()
            
            # Backward pass
            beta = np.zeros((T_eff, self.K))
            beta[-1] = 1 / scaling[-1]
            
            for t in range(T_eff-2, -1, -1):
                for i in range(self.K):
                    beta[t, i] = sum(self.A[i, j] * 
                                    self._emission_prob(y_current[t+1], y_lagged[t+1], j) * 
                                    beta[t+1, j] for j in range(self.K))
                beta[t] /= scaling[t]
            
            # Compute gamma
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True)
            
            # M-step
            self.pi = gamma[0]
            
            # Update A
            for i in range(self.K):
                for j in range(self.K):
                    numer = sum(alpha[t,i] * self.A[i,j] * 
                               self._emission_prob(y_current[t+1], y_lagged[t+1], j) * 
                               beta[t+1,j] for t in range(T_eff-1))
                    denom = gamma[:-1, i].sum()
                    self.A[i,j] = numer / (denom + 1e-10)
            self.A /= self.A.sum(axis=1, keepdims=True)
            
            # Update AR parameters (weighted least squares)
            for k in range(self.K):
                weights = gamma[:, k]
                weight_sum = weights.sum()
                
                if weight_sum > 0:
                    # Construct design matrix: [1, y_{t-1}, ..., y_{t-p}]
                    X = np.column_stack([np.ones(T_eff), y_lagged])
                    W = np.diag(weights)
                    
                    # Weighted least squares
                    XtWX = X.T @ W @ X
                    XtWy = X.T @ W @ y_current
                    
                    try:
                        theta = np.linalg.solve(XtWX + 1e-6*np.eye(self.p+1), XtWy)
                        self.phi0[k] = theta[0]
                        self.phi[k] = theta[1:]
                    except:
                        pass  # Keep previous values if singular
                    
                    # Update variance
                    residuals = y_current - (self.phi0[k] + y_lagged @ self.phi[k])
                    self.sigma[k] = np.sqrt((weights * residuals**2).sum() / weight_sum)
            
            if verbose and (iteration % 10 == 0 or iteration == max_iter-1):
                print(f'Iter {iteration:3d}: log L = {log_lik:.4f}')
            
        return self
    
    def predict_states(self, y):
        # Viterbi decoding
        T = len(y)
        y_lagged = np.zeros((T-self.p, self.p))
        for i in range(self.p):
            y_lagged[:, i] = y[self.p-i-1:T-i-1]
        y_current = y[self.p:]
        T_eff = len(y_current)
        
        log_delta = np.zeros((T_eff, self.K))
        psi = np.zeros((T_eff, self.K), dtype=int)
        
        # Initialize
        for k in range(self.K):
            log_delta[0, k] = np.log(self.pi[k] + 1e-10) + \
                             np.log(self._emission_prob(y_current[0], y_lagged[0], k) + 1e-10)
        
        # Forward
        for t in range(1, T_eff):
            for j in range(self.K):
                probs = log_delta[t-1] + np.log(self.A[:, j] + 1e-10)
                psi[t, j] = probs.argmax()
                log_delta[t, j] = probs[psi[t, j]] + \
                                 np.log(self._emission_prob(y_current[t], y_lagged[t], j) + 1e-10)
        
        # Backtrack
        states = np.zeros(T_eff, dtype=int)
        states[-1] = log_delta[-1].argmax()
        for t in range(T_eff-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states

print('MS-AR class defined')
print('\nFitting MS-AR(1) model...')
msar = MSAR(n_states=2, ar_order=1)
msar.fit(y, max_iter=50, verbose=True)

print('\nLearned parameters:')
for k in range(2):
    print(f'\nState {k}:')
    print(f'  phi0: {msar.phi0[k]:.6f}')
    print(f'  phi1: {msar.phi[k,0]:.6f}')
    print(f'  sigma: {msar.sigma[k]:.6f}')

decoded = msar.predict_states(y)
print(f'\nDecoded {len(decoded)} states')

## Exercise 1: AR Coefficient Interpretation

**Task**: Analyze regime-dependent dynamics:
1. Compute half-life of mean reversion for each state
2. Identify persistent vs. mean-reverting regimes
3. Compare to true parameters

**Expected Output**: Dynamics analysis by regime

In [ ]:
# YOUR CODE HERE
def analyze_ar_dynamics(msar):
    # Compute half-life: -log(2) / log(phi1)
    # Persistence measure
    return None

dynamics = analyze_ar_dynamics(msar)

In [ ]:
def test_exercise_1():
    assert dynamics is not None
    print('✅ Exercise 1 passed!')
test_exercise_1()

## Exercise 2: Multi-Step Forecasting

**Task**: Implement regime-conditional forecasting:
1. Forecast h steps ahead given current regime
2. Forecast with regime uncertainty (mixture)
3. Compute forecast intervals

**Expected Output**: Forecasts and intervals

In [ ]:
# YOUR CODE HERE
def forecast_msar(msar, y_history, current_state, horizon=10):
    # Multi-step ahead forecast
    return None

forecasts = forecast_msar(msar, y[-5:], decoded[-1], horizon=10)

In [ ]:
def test_exercise_2():
    assert forecasts is not None
    print('✅ Exercise 2 passed!')
test_exercise_2()

## Exercise 3: Model Comparison

**Task**: Compare MS-AR vs. standard AR:
1. Fit standard AR(1) to same data
2. Compute in-sample fit (AIC/BIC)
3. Out-of-sample forecast accuracy

**Expected Output**: Comparison table

In [ ]:
# YOUR CODE HERE
def compare_msar_vs_ar(y, msar_model):
    # Fit AR(1) and compare
    return None

comparison = compare_msar_vs_ar(y, msar)

In [ ]:
def test_exercise_3():
    assert comparison is not None
    print('✅ Exercise 3 passed!')
test_exercise_3()

## Summary

### Key Takeaways
1. MS-AR models combine HMM with autoregressive dynamics
2. Regime-dependent coefficients capture changing persistence
3. Useful for financial series with structural breaks
4. EM algorithm extends naturally to AR emissions
5. Better forecasting under regime switching

### Financial Applications
- Interest rate modeling (persistent vs. volatile regimes)
- Exchange rates (trending vs. mean-reverting)
- Commodity prices (storage arbitrage regimes)
- Credit spreads (crisis vs. normal)

### Extensions
- MS-GARCH: Regime-switching volatility
- MS-VAR: Multivariate systems
- MS-ARIMA: Include differencing and MA terms

---